In [1]:
# ==========================================
# CELL 1: Imports & Device Configuration
# ==========================================
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Set up the GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Active Compute Device: {device}")

🚀 Active Compute Device: cuda


In [2]:
# ==========================================
# CELL 2: Custom CelebA Dataset
# ==========================================
class CelebADataset(Dataset):
    def __init__(self, data_dir, attr_file_path, transform=None):
        self.transform = transform
        self.image_files = []
        self.labels = {} 

        if not os.path.exists(data_dir):
            raise FileNotFoundError(f"Directory not found: {data_dir}")
        if not os.path.exists(attr_file_path):
            raise FileNotFoundError(f"Attribute file not found: {attr_file_path}")

        print(f"Parsing attributes from {attr_file_path}...")
        self._parse_attributes(attr_file_path)

        print(f"Scanning '{data_dir}' for images...")
        for root, _, files in os.walk(data_dir):
            for file in files:
                if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    if file in self.labels:
                        self.image_files.append(os.path.join(root, file))
        
        print(f"✅ Successfully linked {len(self.image_files)} images with labels.")

    def _parse_attributes(self, attr_file_path):
        with open(attr_file_path, 'r') as f:
            lines = f.readlines()

        if ',' in lines[0]:
            headers = lines[0].strip().split(',')
            data_start = 1
            delimiter = ','
            offset = 0
        else:
            headers = lines[1].strip().split()
            data_start = 2
            delimiter = None
            offset = 1 

        idx_male = headers.index("Male")
        idx_black = headers.index("Black_Hair")
        idx_blond = headers.index("Blond_Hair")
        idx_brown = headers.index("Brown_Hair")
        idx_gray = headers.index("Gray_Hair")
        idx_young = headers.index("Young")

        for line in lines[data_start:]:
            if not line.strip(): continue
            parts = line.strip().split(delimiter)
            filename = parts[0]
            
            gender = 1.0 if parts[idx_male + offset] == '1' else 0.0
            black = 1.0 if parts[idx_black + offset] == '1' else 0.0
            blond = 1.0 if parts[idx_blond + offset] == '1' else 0.0
            brown = 1.0 if parts[idx_brown + offset] == '1' else 0.0
            gray = 1.0 if parts[idx_gray + offset] == '1' else 0.0
            age = 1.0 if parts[idx_young + offset] == '1' else 0.0

            # Store as a standard Python list to save memory
            self.labels[filename] = [gender, black, blond, brown, gray, age]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx] 
        filename = os.path.basename(img_path) 
        
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
            
        # Convert list to tensor at the exact moment of extraction
        label_data = self.labels[filename]
        label = torch.tensor(label_data, dtype=torch.float32)
        return image, label

def get_data_loader(data_dir, attr_file_path, batch_size=32):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    dataset = CelebADataset(data_dir, attr_file_path, transform=transform)
    # WARNING: num_workers must be 0 in Jupyter on Windows!
    return DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True, num_workers=0)

In [3]:
# ==========================================
# CELL 3: Multitask MobileNetV2 Model
# ==========================================
class MultitaskFaceModel(nn.Module):
    def __init__(self):
        super(MultitaskFaceModel, self).__init__()
        
        # Shared Backbone
        mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        self.backbone = mobilenet.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Classification Heads
        self.gender_head = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(1280, 1),
            nn.Sigmoid()
        )
        
        self.hair_head = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(1280, 4), 
            nn.Sigmoid()
        )

        self.age_head = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(1280, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        
        pred_gender = self.gender_head(x)
        pred_hair = self.hair_head(x)
        pred_age = self.age_head(x)
        
        return pred_gender, pred_hair, pred_age

print("🧠 Architecture Initialized: MultitaskFaceModel")

🧠 Architecture Initialized: MultitaskFaceModel


In [4]:
# ==========================================
# CELL 4: Training Loop with Academic Log
# ==========================================
def analyze_epoch_results(epoch, total_epochs, train_loss):
    """Prints the academic defense text based on the epoch."""
    print("\n" + "="*60)
    print(f"📊 EPOCH {epoch}/{total_epochs} ANALYSIS")
    print(f"   Average Train Loss: {train_loss:.4f}")
    print("-" * 60)
    if epoch <= 3:
        print("🧠 PHASE 1: Random Guessing. High BCE penalty expected.")
    elif 3 < epoch <= 8:
        print("📐 PHASE 2: Topological Feature Discovery (Convex Hull mapping).")
    elif 8 < epoch <= 15:
        print("🔗 PHASE 3: Multitask Regularization and textural extraction.")
    else:
        print("✅ PHASE 4: Mathematical Convergence.")
    print("="*60 + "\n")

# --- FILE PATHS (UPDATE THESE) ---
data_path = r"D:\Imagen Classes UPV\Project\data\raw"
attr_path = r"D:\Imagen Classes UPV\Project\data\raw\list_attr_celeba.txt" 
save_path = r"D:\Imagen Classes UPV\Project\models\custom_face_model_v3.pth"

# Initialize
loader = get_data_loader(data_path, attr_path, batch_size=32)
model = MultitaskFaceModel().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss() 

num_epochs = 15

for epoch in range(1, num_epochs + 1):
    print(f"\n🚀 Starting Epoch {epoch}/{num_epochs}...")
    model.train()
    epoch_loss = 0.0
    batches_processed = 0

    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        
        pred_gender, pred_hair, pred_age = model(data)
        
        loss_gender = criterion(pred_gender, target[:, 0:1]) 
        loss_hair = criterion(pred_hair, target[:, 1:5]) 
        loss_age = criterion(pred_age, target[:, 5:6]) 
        
        total_loss = loss_gender + loss_hair + loss_age
        total_loss.backward()
        optimizer.step()
        
        epoch_loss += total_loss.item()
        batches_processed += 1
        
        if batch_idx % 10 == 0:
            print(f"   Batch {batch_idx} | Total Loss: {total_loss.item():.4f}")
            

    avg_train_loss = epoch_loss / batches_processed
    analyze_epoch_results(epoch, num_epochs, avg_train_loss)

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    torch.save(model.state_dict(), save_path)
    print(f"✅ Model saved to {save_path}")

Parsing attributes from D:\Imagen Classes UPV\Project\data\raw\list_attr_celeba.txt...
Scanning 'D:\Imagen Classes UPV\Project\data\raw' for images...
✅ Successfully linked 202599 images with labels.

🚀 Starting Epoch 1/15...
   Batch 0 | Total Loss: 2.0639
   Batch 10 | Total Loss: 0.8310
   Batch 20 | Total Loss: 1.0598
   Batch 30 | Total Loss: 0.8579
   Batch 40 | Total Loss: 0.8162
   Batch 50 | Total Loss: 0.8414
   Batch 60 | Total Loss: 0.8663
   Batch 70 | Total Loss: 0.8067
   Batch 80 | Total Loss: 0.5577
   Batch 90 | Total Loss: 0.7525
   Batch 100 | Total Loss: 0.9459
   Batch 110 | Total Loss: 0.7093
   Batch 120 | Total Loss: 0.7034
   Batch 130 | Total Loss: 0.7069
   Batch 140 | Total Loss: 0.5928
   Batch 150 | Total Loss: 0.7738
   Batch 160 | Total Loss: 0.8825
   Batch 170 | Total Loss: 0.8797
   Batch 180 | Total Loss: 0.7838
   Batch 190 | Total Loss: 0.7413
   Batch 200 | Total Loss: 0.7861
   Batch 210 | Total Loss: 0.5738
   Batch 220 | Total Loss: 0.6140
   